# Agent Frameworks：从底层原理到 PydanticAI、LangGraph 与 Microsoft Agent Framework

**三秒结论：Agent 框架负责组织模型、工具、状态和执行流程。选择框架时，先看控制权、状态归属、暂停恢复和副作用边界。**

本章先完整设计一条“共享测试环境部署申请”流程，再分别实现：

1. **PydanticAI**：读取证据，输出类型化准入结论。
2. **LangGraph**：执行准入、暂停审批、恢复和测试任务入队。
3. **Microsoft Agent Framework**：用 Executor、人工请求和 Workflow 完成同一流程。

实验全部在 Notebook 本地运行。PR、CI、测试环境和部署队列由内存数据模拟，不要求安装 GitLab、Jenkins、Kubernetes 或审批系统。


## 1 结论与学习目标

学完本章，你需要理解以下内容：

1. Agent 框架的本质、产生背景和底层共同结构。
2. 模型驱动、工作流驱动和混合控制三种编程模型。
3. 多 Agent、Workflow、Harness、MCP 和应用平台之间的关系。
4. PydanticAI、LangGraph 与 Microsoft Agent Framework 各自擅长解决什么问题。
5. 同一任务在三个框架中的数据流、状态流和恢复路径。
6. 如何根据任务风险、流程稳定性、团队能力和运维条件选择框架。


### 1.1 先记住六个判断

1. **框架减少重复工程，不提高模型智力。** 模型是否会推理仍由模型、上下文和工具反馈决定。
2. **框架的主要差异是控制流和状态归属。** API 命名只是表面差异。
3. **多 Agent 属于任务拓扑层。** 一个 Workflow 可以包含一个 Agent，也可以包含多个 Agent。
4. **结构化输出成功不等于业务正确。** 类型校验只能证明字段和类型符合约定。
5. **Checkpoint 保存可恢复状态，不自动撤销外部副作用。** 写数据库、发消息和创建工单仍需幂等设计。
6. **先用最小抽象完成任务，再增加框架能力。** 固定流程优先使用普通代码或 Workflow；开放任务才需要模型自主循环。


## 2 背景：框架为什么产生

### 2.1 从一次模型调用到 Agent 应用

最小模型调用只处理“输入消息 → 输出消息”。一个可持续运行的 Agent 还需要处理：

| 工程问题 | 直接手写时需要实现的内容 |
| --- | --- |
| 工具调用 | Schema、注册表、参数校验、分发、结果回传 |
| 结构化结果 | 输出约束、解析、校验、失败重试 |
| 多步骤执行 | 循环、路由、停止条件、错误分支 |
| 状态 | 会话状态、业务状态、Checkpoint、恢复 |
| 人工介入 | 暂停、审批请求、恢复令牌、拒绝路径 |
| 可观测性 | 消息、工具调用、节点事件、耗时和错误 |
| 多 Agent | 角色、交接、共享状态、并行和汇合 |

第 4、7、8、10 章已经分别实现过这些机制。框架把反复出现的结构做成标准接口和运行时。[1]


### 2.2 定义：Agent Framework 是什么

**Agent Framework 是用于定义 Agent、连接模型与工具、编排控制流并管理运行状态的软件库。**

一个框架通常包含三部分：

1. **编程模型**：用 Agent、Node、Edge、Workflow、Handoff 等概念描述任务。
2. **运行时**：负责调度、暂停、恢复、重试、流式事件和并发。
3. **集成层**：连接模型供应商、工具协议、存储、Tracing 和部署环境。

框架提供默认实现和扩展点。业务规则、权限边界、幂等键、验收标准和数据治理仍由应用负责。


### 2.3 从第一性原理拆出框架内核

```mermaid
flowchart TD
    A["Agent 定义<br/>指令、模型、工具、输出"] --> R["Runner 或 Scheduler"]
    U["用户输入"] --> R
    R --> C["Context Builder"]
    C --> M["Model Adapter"]
    M --> R
    R --> T["Tool Registry / Executor"]
    T --> R
    R --> S["State / Session"]
    R --> E["Events / Trace"]
    R --> K["Checkpoint"]
    R --> O["最终输出或中断请求"]
```

不同框架都要回答五个底层问题：

1. 下一步由模型决定，还是由程序中的边和条件决定？
2. 状态保存在消息历史、状态对象、会话还是外部存储？
3. 工具调用失败后，谁决定重试、回退或终止？
4. 任务暂停后，用什么标识找到原运行并继续？
5. 框架升级后，旧状态和旧 Checkpoint 是否还能读取？


### 2.4 框架、SDK、Harness 和应用平台的层次

| 层次 | 主要内容 | 典型例子 |
| --- | --- | --- |
| Model API | 消息、生成、Tool Call、流式响应 | OpenAI-compatible API |
| Agent SDK | Agent、工具、类型化输出、会话 | PydanticAI、OpenAI Agents SDK |
| Graph / Workflow Runtime | 状态、节点、边、中断、恢复 | LangGraph、Microsoft Agent Framework Workflow |
| Harness | 环境、权限、事件、Artifact、预算、验收 | 第 10 章实现的运行系统 |
| Application Platform | UI、知识库、工作流配置、运营和部署 | Dify、Coze Studio |

这些层次允许重叠。LangGraph 既有图运行时，也通过 LangChain 生态连接模型和工具；Microsoft Agent Framework 同时提供 Agent SDK 和 Workflow。Dify 与 Coze Studio 面向完整应用交付，比较维度已经超出 Python 框架。[14][15]


## 3 分类：先看控制权，再看品牌

### 3.1 三种控制模型

```mermaid
flowchart LR
    subgraph A["模型驱动"]
        A1["输入"] --> A2["模型选择工具或结束"]
        A2 --> A3["工具结果"]
        A3 --> A2
    end
    subgraph B["工作流驱动"]
        B1["状态"] --> B2["节点"]
        B2 --> B3["程序条件路由"]
        B3 --> B4["下一节点"]
    end
    subgraph C["混合控制"]
        C1["Workflow"] --> C2["Agent 节点"]
        C2 --> C3["工具循环"]
        C3 --> C4["Workflow 路由"]
    end
```

| 模式 | 下一步主要由谁决定 | 优点 | 代价 |
| --- | --- | --- | --- |
| 模型驱动 | 模型 | 灵活，适合开放任务 | 路径难预测，成本和失败面更大 |
| 工作流驱动 | 程序条件或图 | 路径清晰，容易恢复和审计 | 流程变化时需要改图或代码 |
| 混合控制 | Workflow 管大阶段，Agent 管局部探索 | 兼顾确定性和灵活性 | 两套状态与错误语义需要对齐 |


### 3.2 多 Agent 属于组合方式

多 Agent 系统至少包含两个拥有独立指令、工具或上下文的 Agent。常见拓扑如下：

| 拓扑 | 数据流 | 适用条件 | 主要风险 |
| --- | --- | --- | --- |
| 顺序交接 | A → B → C | 阶段边界明确 | 信息在交接中丢失 |
| Supervisor | 主管 → 专家 → 主管 | 专家工具差异明显 | 主管成为瓶颈 |
| 并行汇总 | A、B 并行 → 汇总 | 子任务互相独立 | 重复上下文和 Token 成本 |
| 群聊 | 多角色轮流发言 | 需要观点碰撞 | 终止困难、对话膨胀 |

同一个模型配置出三个角色，不会自动产生更高质量。拆分需要满足至少一项：权限不同、工具不同、上下文隔离、并行收益明确、验收职责独立。


## 4 市场扫描与选型方法

### 4.1 先按技术类别筛选

本章不追求覆盖全部框架。筛选标准如下：

1. 有持续维护的官方仓库和文档。
2. 能表达该类别最有代表性的底层能力。
3. Python API 可以运行最小实验和生产关键路径。
4. 支持 OpenAI-compatible Endpoint，或存在可解释的适配方式。
5. 能讨论状态、恢复、人审、可观测性和升级风险。
6. 与其他入选框架的教学价值不重复。

活跃度只能证明项目仍有人维护。Star 数、下载量和社交热度不能证明它适合具体系统。


### 4.2 候选框架与本章选择

| 类别 | 候选 | 本章选择 | 选择原因 |
| --- | --- | --- | --- |
| 类型化 Agent SDK | PydanticAI、OpenAI Agents SDK、Google ADK、Strands Agents | **PydanticAI** | Python 类型、依赖注入、输出协议和测试模型集中在一个小实验中展示 [2][9][10][11] |
| 状态图 / Workflow | LangGraph、LlamaIndex Workflows、CrewAI Flows | **LangGraph** | 显式 State、Edge、Checkpoint、Interrupt 最适合解释恢复语义 [4][12][13] |
| 企业 Agent + Workflow | Microsoft Agent Framework、AutoGen、Semantic Kernel | **Microsoft Agent Framework** | 官方后继路线同时覆盖 Agent、Workflow、请求响应和多 Agent [6] |
| 应用平台 | Dify、Coze Studio | 仅说明边界 | 重点是可视化应用交付、运营和部署，无法只用 Python 框架维度公平比较 [14][15] |

OpenAI Agents SDK 强调 Agent、Handoff、Guardrail、Session 和 Tracing，适合围绕 OpenAI 模型与轻量多 Agent 交接构建应用。[9] Google ADK 与 Strands Agents 也保持活跃，本章把有限篇幅留给底层机制差异更明显的三个实现。[10][11]


### 4.3 AutoGen 与 Semantic Kernel 放在哪里

Microsoft Agent Framework 是 Microsoft 对 AutoGen 与 Semantic Kernel Agent 能力的后继整合路线。官方文档将其描述为两个项目的直接后继，并提供 Agent 与图式 Workflow 两套能力。[6]

因此本章不再分别实现旧的 AutoGen GroupChat 和 Semantic Kernel Agent。迁移时仍需关注：

1. 已有 AutoGen / Semantic Kernel 应用不会自动变成新 Workflow。
2. 消息、插件、线程、序列化状态和人工请求的 API 需要逐项迁移。
3. 生产系统应先锁定版本并验证 Checkpoint 兼容，再升级运行时。


## 5 统一实验：共享测试环境部署申请

### 5.1 任务目标与边界

开发人员希望把 PR 对应的代码部署到共享测试环境，供产品、研发和测试联合验收。Agent 完成以下工作：

1. 读取 PR、CI 和目标环境证据。
2. 判断当前变更是否具备测试部署条件。
3. 数据库迁移等高风险变更等待 Team Leader 批准。
4. 准入通过后创建一条状态为 `QUEUED` 的测试部署任务。

实验到“任务入队”结束。部署动作由 CI/CD Runner 执行，Agent 不直接登录服务器，也不持有生产环境凭据。


### 5.2 先区分四个经常混用的动作

| 动作 | 实际含义 | 通常执行者 | 本章是否执行 |
| --- | --- | --- | --- |
| 合并代码 | PR 合并到受保护分支 | 代码平台按权限执行 | 否 |
| 触发流水线 | 向 CI/CD 系统创建一次 Job | 用户、Agent 或平台服务账号 | 本地模拟 |
| 执行部署 | Runner 拉取制品并更新目标环境 | CI/CD Runner | 否 |
| 生产发布 | 经过变更审批后部署生产制品 | 运维或发布平台 | 否 |

本章中的“创建测试部署任务”只对应第二行。真实系统中，它可能是调用 GitLab Pipeline、GitHub Actions、Jenkins 或内部发布平台 API。API 返回“已入队”，不代表部署成功。


### 5.3 角色、权限与责任

| 参与者 | 可以做什么 | 不应做什么 |
| --- | --- | --- |
| 开发人员 | 发起测试部署申请、查看结果 | 绕过 CI 和审批规则 |
| Agent | 读取证据、生成建议、请求审批、创建测试任务 | 合并 PR、持有生产凭据、直接执行生产部署 |
| Team Leader | 批准数据库迁移等共享环境风险 | 代替 Runner 执行部署 |
| CI/CD Runner | 使用受控凭据执行测试环境部署 | 自行改变准入结论 |
| 运维 / 发布管理员 | 管理环境、凭据和生产发布 | 把生产凭据交给 Agent |

生产部署需要单独的变更单、审批链、发布窗口、回滚演练和生产权限。本章没有注册任何生产部署工具，因此模型即使提出生产发布，也没有可调用的执行入口。


### 5.4 本地模拟器与真实系统的对应关系

| Notebook 对象 | 本地实现 | 生产系统映射 |
| --- | --- | --- |
| PR 仓库 | Python 字典 | GitHub / GitLab PR API |
| CI 报告仓库 | Python 字典 | CI API、测试报告服务 |
| 环境注册表 | Python 字典 | CMDB、环境管理平台 |
| 审批结果 | Notebook 中注入对象 | 审批平台回调、消息卡片 |
| 测试部署队列 | Python 字典 | CI/CD Pipeline API |

两者使用相同的数据协议和流程边界。教学环境替换了 I/O 适配器，没有删掉权限、审批、幂等和失败处理。读者从上到下执行 Notebook 即可看到完整流程。


### 5.5 验收标准

实验必须同时满足以下条件：

1. CI 失败、CI 对应旧 Commit、目标环境被占用时，流程返回 `blocked`。
2. 变更包含数据库迁移时，流程返回 `approval_required`。
3. 人工批准前，部署队列为空。
4. 人工拒绝后，部署队列仍为空。
5. 人工批准后，只创建一条 `queued` 任务。
6. 同一 PR、Commit 和环境重复入队时返回同一任务，不产生重复部署。
7. Agent 的工具协议中不存在生产部署能力。


### 5.6 实现过程与完整时序

```mermaid
sequenceDiagram
    participant D as 开发人员
    participant A as Agent Workflow
    participant E as 本地证据仓库
    participant L as Team Leader
    participant Q as 测试部署队列
    participant R as CI/CD Runner

    D->>A: 申请部署 PR-1842 到 integration-test-2
    A->>E: 读取 PR、CI、环境状态
    E-->>A: 返回同一 Commit 的证据
    A->>A: 执行确定性准入规则
    alt 证据不完整或规则失败
        A-->>D: blocked + 原因
    else 普通变更
        A->>Q: 使用幂等键创建测试任务
        Q-->>A: queued
    else 包含数据库迁移
        A-->>L: 请求审批
        L-->>A: 批准或拒绝
        alt 批准
            A->>Q: 使用幂等键创建测试任务
            Q-->>A: queued
        else 拒绝
            A-->>D: rejected
        end
    end
    Q-->>R: 真实系统中由 Runner 消费
```

流程把“判断”和“执行”分开。模型可以解释风险，准入规则、审批状态和入队条件由确定性代码检查。


### 5.7 失败发生在哪里，流程跳到哪里

| 失败点 | 例子 | 下一状态 | 是否允许入队 |
| --- | --- | --- | --- |
| 读取证据 | CI 报告不存在 | `blocked` | 否 |
| 证据一致性 | CI 的 Commit SHA 与 PR 不同 | `blocked` | 否 |
| CI 质量门 | 测试失败 | `blocked` | 否 |
| 环境检查 | 共享环境被其他服务占用 | `blocked` | 否 |
| 人工审批 | Team Leader 拒绝 | `rejected` | 否 |
| 队列调用 | 请求超时、结果未知 | 查询幂等键后重试 | 视回执而定 |
| Runner 执行 | 构建或部署失败 | `queued` 之后的外部状态 | 本章不处理 |

`queued` 只证明准入完成且任务已创建。后续还可能出现 `running`、`succeeded`、`failed` 或 `cancelled`，这些状态属于 CI/CD 系统。


### 5.8 开发卡点、涉及知识与框架应对

| 开发卡点 | 涉及知识 | PydanticAI | LangGraph | Microsoft Agent Framework |
| --- | --- | --- | --- | --- |
| 外部数据如何进入 Agent | 依赖注入、Tool Calling | `deps_type` + Tool | Node 读取 State / Repository | Tool 或 Executor |
| 输出字段必须稳定 | Pydantic、Schema 校验 | `output_type` 原生支持 | 节点边界主动校验 | 类型化消息或 Provider 输出 |
| 分支必须可预测 | 规则引擎、控制流 | 在 Agent 外部路由 | Conditional Edge | Conditional Edge |
| 审批等待数分钟或数小时 | 中断、关联 ID、恢复 | 组合外部运行时 | `interrupt` + Checkpoint | `request_info` + Response |
| 恢复后不得重复入队 | 幂等、回执、故障窗口 | 工具层负责 | Node + 业务幂等键 | Executor + 业务幂等键 |
| 进程重启后继续 | 持久化、状态版本 | Durable Execution 集成 | 数据库 Checkpointer | Workflow Checkpoint |
| 排查一次错误运行 | Tracing、事件协议 | 模型消息与 Tool Part | State History 与节点事件 | Workflow Event |

框架负责表达和调度这些机制。准入规则、权限、幂等键和数据归属由业务系统定义。


### 5.9 三个框架如何保持可比

三个实验共享以下内容：

1. 同一个 PR：`PR-1842`。
2. 同一份 CI 和环境证据。
3. 同一个 `assess_readiness()` 准入函数。
4. 同一个 `queue_test_deployment()` 幂等入队函数。
5. 同一条高风险规则：包含 `migrations/` 时等待人工审批。

比较对象只有框架的表达方式：PydanticAI 观察工具与类型化输出；LangGraph 观察 State、Edge、Interrupt 和恢复；Microsoft Agent Framework 观察 Executor、请求响应和 Workflow。


In [1]:
import os
from dotenv import find_dotenv, load_dotenv

# 从仓库根目录查找 .env；进程中显式设置的环境变量优先。
ENV_PATH = find_dotenv(usecwd=True)
if ENV_PATH:
    load_dotenv(ENV_PATH, override=False)

RUN_LIVE_LLM = os.getenv("RUN_LIVE_LLM", "0") == "1"
LLM_API_KEY = (
    os.getenv("LLM_API_KEY")
    or os.getenv("DEEPSEEK_API_KEY")
    or os.getenv("OPENAI_API_KEY")
    or ""
).strip()
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "https://api.deepseek.com").strip()
LLM_MODEL = os.getenv("LLM_MODEL", "deepseek-chat").strip()

print({
    "run_live_llm": RUN_LIVE_LLM,
    "api_key_configured": bool(LLM_API_KEY),
    "base_url": LLM_BASE_URL,
    "model": LLM_MODEL,
})


{'run_live_llm': False, 'api_key_configured': True, 'base_url': 'https://api.deepseek.com', 'model': 'deepseek-v4-pro'}


真实模型实验继续使用全课程统一的环境变量：

```dotenv
RUN_LIVE_LLM=1
LLM_API_KEY=你的密钥
LLM_BASE_URL=https://api.deepseek.com
LLM_MODEL=deepseek-chat
```

本地框架实验不需要 API Key。发布版设置 `RUN_LIVE_LLM=0`，Notebook 只显示 Key 是否配置，不输出 Key。


In [2]:
from typing import Literal
from pydantic import BaseModel, Field


class PullRequest(BaseModel):
    pr_id: str
    commit_sha: str
    service: str
    target_env: Literal["integration-test-1", "integration-test-2"]
    changed_files: list[str]
    ci_report_id: str
    requested_by: str


class CIReport(BaseModel):
    report_id: str
    commit_sha: str
    status: Literal["passed", "failed", "running"]
    failed_tests: int = Field(ge=0)
    coverage: float = Field(ge=0, le=100)


class EnvironmentState(BaseModel):
    environment_id: Literal["integration-test-1", "integration-test-2"]
    status: Literal["available", "occupied"]
    occupied_by: str | None = None


class ReadinessDecision(BaseModel):
    status: Literal["blocked", "ready", "approval_required"]
    reasons: list[str]
    requires_approval: bool
    recommended_action: str


class ApprovalDecision(BaseModel):
    approved: bool
    actor: str
    comment: str


class DeploymentJob(BaseModel):
    job_id: str
    idempotency_key: str
    pr_id: str
    commit_sha: str
    target_env: str
    status: Literal["queued"]
    approved_by: str | None = None


class DeploymentOutcome(BaseModel):
    status: Literal["blocked", "rejected", "queued"]
    reasons: list[str] = Field(default_factory=list)
    job: DeploymentJob | None = None


In [3]:
PR_ID = "PR-1842"

PULL_REQUESTS = {
    PR_ID: PullRequest(
        pr_id=PR_ID,
        commit_sha="a1b2c3d4",
        service="order-service",
        target_env="integration-test-2",
        changed_files=[
            "src/order/repository.py",
            "migrations/20260720_add_order_index.sql",
            "tests/test_order_repository.py",
        ],
        ci_report_id="CI-9001",
        requested_by="developer-li",
    )
}

CI_REPORTS = {
    "CI-9001": CIReport(
        report_id="CI-9001",
        commit_sha="a1b2c3d4",
        status="passed",
        failed_tests=0,
        coverage=88.5,
    )
}

ENVIRONMENTS = {
    "integration-test-1": EnvironmentState(
        environment_id="integration-test-1",
        status="occupied",
        occupied_by="inventory-service",
    ),
    "integration-test-2": EnvironmentState(
        environment_id="integration-test-2",
        status="available",
    ),
}

print(PULL_REQUESTS[PR_ID].model_dump())
print(CI_REPORTS["CI-9001"].model_dump())
print(ENVIRONMENTS["integration-test-2"].model_dump())


{'pr_id': 'PR-1842', 'commit_sha': 'a1b2c3d4', 'service': 'order-service', 'target_env': 'integration-test-2', 'changed_files': ['src/order/repository.py', 'migrations/20260720_add_order_index.sql', 'tests/test_order_repository.py'], 'ci_report_id': 'CI-9001', 'requested_by': 'developer-li'}
{'report_id': 'CI-9001', 'commit_sha': 'a1b2c3d4', 'status': 'passed', 'failed_tests': 0, 'coverage': 88.5}
{'environment_id': 'integration-test-2', 'status': 'available', 'occupied_by': None}


In [4]:
def assess_readiness(
    pr: PullRequest,
    report: CIReport,
    environment: EnvironmentState,
) -> ReadinessDecision:
    '''根据证据执行确定性测试部署准入规则。'''
    if report.commit_sha != pr.commit_sha:
        return ReadinessDecision(
            status="blocked",
            reasons=["CI 报告对应旧 Commit"],
            requires_approval=False,
            recommended_action="重新运行当前 Commit 的 CI",
        )
    if report.status != "passed" or report.failed_tests > 0:
        return ReadinessDecision(
            status="blocked",
            reasons=["CI 未通过"],
            requires_approval=False,
            recommended_action="修复失败测试并重新运行 CI",
        )
    if environment.environment_id != pr.target_env:
        return ReadinessDecision(
            status="blocked",
            reasons=["环境证据与目标环境不一致"],
            requires_approval=False,
            recommended_action="重新读取目标环境状态",
        )
    if environment.status != "available":
        return ReadinessDecision(
            status="blocked",
            reasons=[f"目标环境被 {environment.occupied_by} 占用"],
            requires_approval=False,
            recommended_action="选择空闲环境或等待释放",
        )

    has_migration = any(path.startswith("migrations/") for path in pr.changed_files)
    if has_migration:
        return ReadinessDecision(
            status="approval_required",
            reasons=["变更包含数据库迁移，共享环境数据可能受影响"],
            requires_approval=True,
            recommended_action="Team Leader 批准后创建测试部署任务",
        )

    return ReadinessDecision(
        status="ready",
        reasons=["CI 通过且目标环境可用"],
        requires_approval=False,
        recommended_action="创建测试部署任务",
    )


baseline = assess_readiness(
    PULL_REQUESTS[PR_ID],
    CI_REPORTS["CI-9001"],
    ENVIRONMENTS["integration-test-2"],
)
print(baseline.model_dump())
assert baseline.status == "approval_required"


{'status': 'approval_required', 'reasons': ['变更包含数据库迁移，共享环境数据可能受影响'], 'requires_approval': True, 'recommended_action': 'Team Leader 批准后创建测试部署任务'}


In [5]:
def queue_test_deployment(
    store: dict[str, dict],
    pr: PullRequest,
    decision: ReadinessDecision,
    approval: ApprovalDecision | None = None,
) -> DeploymentJob:
    '''校验准入结果，并用稳定幂等键创建本地测试部署任务。'''
    if decision.status == "blocked":
        raise ValueError("准入已阻止，不能创建部署任务")
    if decision.requires_approval and not (approval and approval.approved):
        raise PermissionError("该变更需要有效的人工批准")

    key = f"test-deploy:{pr.pr_id}:{pr.commit_sha}:{pr.target_env}"
    if key in store:
        return DeploymentJob.model_validate(store[key])

    job = DeploymentJob(
        job_id=f"JOB-{pr.commit_sha.upper()}",
        idempotency_key=key,
        pr_id=pr.pr_id,
        commit_sha=pr.commit_sha,
        target_env=pr.target_env,
        status="queued",
        approved_by=approval.actor if approval else None,
    )
    store[key] = job.model_dump()
    return job


# 同一个请求执行两次，只产生一条任务。
queue_demo: dict[str, dict] = {}
demo_approval = ApprovalDecision(
    approved=True,
    actor="team-leader-wang",
    comment="已确认迁移仅影响共享测试环境",
)
first_job = queue_test_deployment(
    queue_demo, PULL_REQUESTS[PR_ID], baseline, demo_approval
)
second_job = queue_test_deployment(
    queue_demo, PULL_REQUESTS[PR_ID], baseline, demo_approval
)

print(first_job.model_dump())
assert first_job == second_job
assert len(queue_demo) == 1

# 人工拒绝时，入队函数必须拒绝执行，队列保持为空。
rejected_queue: dict[str, dict] = {}
rejected_approval = ApprovalDecision(
    approved=False,
    actor="team-leader-wang",
    comment="迁移脚本缺少回滚验证",
)
try:
    queue_test_deployment(
        rejected_queue,
        PULL_REQUESTS[PR_ID],
        baseline,
        rejected_approval,
    )
except PermissionError as exc:
    print("拒绝路径：", exc)

assert rejected_queue == {}


{'job_id': 'JOB-A1B2C3D4', 'idempotency_key': 'test-deploy:PR-1842:a1b2c3d4:integration-test-2', 'pr_id': 'PR-1842', 'commit_sha': 'a1b2c3d4', 'target_env': 'integration-test-2', 'status': 'queued', 'approved_by': 'team-leader-wang'}
拒绝路径： 该变更需要有效的人工批准


这两段函数形成三个框架共同使用的业务内核：

- `assess_readiness()` 只判断，不产生外部副作用。
- `queue_test_deployment()` 只接受通过准入的数据，并强制检查人工批准。
- 幂等键由 PR、Commit 和目标环境组成。调用方超时后可以使用同一个键重试。

真实项目只需把字典替换成 Repository 和 CI/CD Client。规则和框架代码无需知道底层来自 HTTP API 还是本地数据。


### 5.10 先验证三个失败分支

下面直接调用业务内核。此时还没有任何框架参与，因此失败结果可以作为后续实现的对照基线。


In [6]:
pr = PULL_REQUESTS[PR_ID]

stale_report = CI_REPORTS["CI-9001"].model_copy(
    update={"commit_sha": "old-commit"}
)
failed_report = CI_REPORTS["CI-9001"].model_copy(
    update={"status": "failed", "failed_tests": 2}
)
occupied_env = ENVIRONMENTS["integration-test-1"].model_copy(
    update={"environment_id": "integration-test-2"}
)

failure_results = {
    "stale_ci": assess_readiness(pr, stale_report, ENVIRONMENTS[pr.target_env]),
    "failed_ci": assess_readiness(pr, failed_report, ENVIRONMENTS[pr.target_env]),
    "occupied_env": assess_readiness(pr, CI_REPORTS["CI-9001"], occupied_env),
}

for name, decision in failure_results.items():
    print(name, "->", decision.status, decision.reasons)

assert all(result.status == "blocked" for result in failure_results.values())


stale_ci -> blocked ['CI 报告对应旧 Commit']
failed_ci -> blocked ['CI 未通过']
occupied_env -> blocked ['目标环境被 inventory-service 占用']


## 6 PydanticAI：用类型契约封装准入分析

### 6.1 本节目标

本节只实现“读取部署证据并生成 `ReadinessDecision`”。审批和任务入队继续由普通业务代码负责。

实现顺序：

1. 用依赖对象提供本地证据仓库。
2. 把证据读取注册为只读工具。
3. 用 `output_type` 约束最终结论。
4. 检查消息轨迹中是否出现 Tool Call 和 Tool Return。

主要卡点是模型可能返回合法结构但违反准入规则。解决方式是让模型负责解释和建议，调用 `assess_readiness()` 复核硬规则。


### 6.2 核心对象与案例映射

| 对象 | 职责 | 本实验映射 |
| --- | --- | --- |
| `Agent` | 组合模型、指令、工具和输出 | 测试部署准入分析器 |
| `deps_type` | 声明本次运行可访问的依赖 | PR、CI、环境仓库与调用轨迹 |
| `RunContext` | 把依赖传给工具 | 读取三份证据 |
| `output_type` | 校验最终结果 | `ReadinessDecision` |
| `TestModel` | 生成可预测的离线调用 | 验证框架协议 |

数据流：Prompt → 模型请求证据工具 → 工具从依赖读取数据 → 模型输出准入结论 → Pydantic 校验字段。


### 6.3 离线实验：工具、依赖与类型化输出

`TestModel` 不调用外部 API。它验证四件事：工具被调用、依赖可访问、结果完成类型校验、消息轨迹完整。它不验证模型判断质量。


In [7]:
from dataclasses import dataclass, field
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.test import TestModel


@dataclass
class AdmissionDependencies:
    pull_requests: dict[str, PullRequest]
    ci_reports: dict[str, CIReport]
    environments: dict[str, EnvironmentState]
    tool_calls: list[str] = field(default_factory=list)


pydantic_test_model = TestModel(
    call_tools=["get_deployment_evidence"],
    custom_output_args=baseline.model_dump(),
)

pydantic_agent = Agent(
    pydantic_test_model,
    deps_type=AdmissionDependencies,
    output_type=ReadinessDecision,
    instructions="读取证据并生成共享测试环境部署准入结论。",
)


@pydantic_agent.tool
def get_deployment_evidence(
    ctx: RunContext[AdmissionDependencies],
) -> dict:
    '''读取本次实验固定 PR 的 PR、CI 和环境证据。'''
    ctx.deps.tool_calls.append(PR_ID)
    current_pr = ctx.deps.pull_requests[PR_ID]
    return {
        "pull_request": current_pr.model_dump(),
        "ci_report": ctx.deps.ci_reports[current_pr.ci_report_id].model_dump(),
        "environment": ctx.deps.environments[current_pr.target_env].model_dump(),
    }


pydantic_deps = AdmissionDependencies(
    pull_requests=PULL_REQUESTS,
    ci_reports=CI_REPORTS,
    environments=ENVIRONMENTS,
)
pydantic_result = await pydantic_agent.run(
    f"评估 {PR_ID} 是否可以部署到共享测试环境。",
    deps=pydantic_deps,
)

print("工具调用：", pydantic_deps.tool_calls)
print("结果类型：", type(pydantic_result.output).__name__)
print("准入结论：", pydantic_result.output.model_dump())

assert pydantic_deps.tool_calls == [PR_ID]
assert isinstance(pydantic_result.output, ReadinessDecision)
assert pydantic_result.output.status == "approval_required"


工具调用： ['PR-1842']
结果类型： ReadinessDecision
准入结论： {'status': 'approval_required', 'reasons': ['变更包含数据库迁移，共享环境数据可能受影响'], 'requires_approval': True, 'recommended_action': 'Team Leader 批准后创建测试部署任务'}


In [8]:
# 展开框架消息，观察 Tool Calling Loop 仍然存在。
pydantic_trace = [
    type(part).__name__
    for message in pydantic_result.all_messages()
    for part in message.parts
]

print(pydantic_trace)
assert "ToolCallPart" in pydantic_trace
assert "ToolReturnPart" in pydantic_trace


['UserPromptPart', 'ToolCallPart', 'ToolReturnPart', 'ToolCallPart', 'ToolReturnPart']


输出中的 `ToolCallPart` 和 `ToolReturnPart` 证明框架封装了 Tool Calling 协议。最终结果已经是 `ReadinessDecision` 对象。

业务代码仍需执行硬规则复核：

```python
policy_decision = assess_readiness(pr, report, environment)
```

当模型结论与策略结论冲突时，以策略结论为准，并记录冲突供 Evals 使用。


### 6.4 结构化输出的三种实现

| 输出方式 | 大致机制 | 优点 | 兼容风险 |
| --- | --- | --- | --- |
| Tool Output | 把最终结果声明为特殊工具 | 校验严格，可自动重试 | 部分兼容服务不支持强制 `tool_choice` |
| Native Output | 使用供应商原生 JSON Schema | 协议清晰 | 依赖模型与 Endpoint 能力 |
| Prompted Output | Schema 进入 Prompt，再本地校验 | 兼容面较广 | 首次输出可能不合格 |

DeepSeek 等 OpenAI-compatible 服务需要逐项验证高级参数。本章真实路径使用 `PromptedOutput`，降低强制工具选择的兼容风险。


### 6.5 可选真实实验：DeepSeek + PromptedOutput

真实模型必须先调用证据工具，再输出 `ReadinessDecision`。发布版默认跳过；本地框架实验不受影响。


In [9]:
from pydantic_ai import PromptedOutput
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider


pydantic_live_output = None

if RUN_LIVE_LLM and LLM_API_KEY:
    pydantic_model = OpenAIChatModel(
        LLM_MODEL,
        provider=OpenAIProvider(
            base_url=LLM_BASE_URL,
            api_key=LLM_API_KEY,
        ),
    )
    pydantic_live_agent = Agent(
        pydantic_model,
        deps_type=AdmissionDependencies,
        output_type=PromptedOutput(ReadinessDecision),
        instructions=(
            "你是共享测试环境准入分析器。必须读取部署证据。"
            "CI 失败、Commit 不一致或环境被占用时 blocked；"
            "数据库迁移需要人工批准。"
        ),
    )

    @pydantic_live_agent.tool
    def get_live_deployment_evidence(
        ctx: RunContext[AdmissionDependencies],
    ) -> dict:
        '''读取固定实验 PR 的三份证据。'''
        ctx.deps.tool_calls.append(PR_ID)
        current_pr = ctx.deps.pull_requests[PR_ID]
        return {
            "pull_request": current_pr.model_dump(),
            "ci_report": ctx.deps.ci_reports[current_pr.ci_report_id].model_dump(),
            "environment": ctx.deps.environments[current_pr.target_env].model_dump(),
        }

    live_deps = AdmissionDependencies(
        PULL_REQUESTS, CI_REPORTS, ENVIRONMENTS
    )
    live_result = await pydantic_live_agent.run(
        f"评估 {PR_ID} 的测试部署准入状态。",
        deps=live_deps,
    )
    pydantic_live_output = live_result.output
    print(pydantic_live_output.model_dump())

    policy_decision = assess_readiness(
        PULL_REQUESTS[PR_ID],
        CI_REPORTS["CI-9001"],
        ENVIRONMENTS["integration-test-2"],
    )
    assert live_deps.tool_calls
    assert pydantic_live_output.status == policy_decision.status
else:
    print("已跳过真实 PydanticAI 调用；设置 RUN_LIVE_LLM=1 后执行。")


已跳过真实 PydanticAI 调用；设置 RUN_LIVE_LLM=1 后执行。


### 6.6 本框架解决了什么，还缺什么

PydanticAI 已处理工具注册、依赖注入、消息循环和输出校验。当前实验仍缺少持久化审批和跨进程恢复。

生产取舍：

1. 请求内完成的类型化 Agent：直接使用 `Agent`。
2. 等待数小时的审批：接入 Durable Execution 或外部 Workflow。[3]
3. 固定准入规则：保留在普通代码或策略系统中。
4. 入队写操作：继续使用业务幂等键和回执查询。


## 7 LangGraph：显式表达完整准入流程

### 7.1 本节目标

本节把同一个案例组织成状态图，完整运行“读取证据 → 准入判断 → 暂停审批 → 恢复 → 幂等入队”。

实现时需要解决四个问题：

1. State 保存哪些业务数据。
2. 哪些分支由确定性条件控制。
3. 审批暂停后如何找到原运行。
4. 恢复或重试时如何避免重复创建任务。

LangGraph 的 State、Conditional Edge、Interrupt 和 Checkpointer 分别对应这四个问题。[4][5]


### 7.2 节点、状态与失败去向

```mermaid
flowchart TD
    S["START"] --> L["load_evidence"]
    L --> A["assess"]
    A --> D{"decision.status"}
    D -- "blocked" --> B["finish_blocked"]
    D -- "ready" --> Q["queue_job"]
    D -- "approval_required" --> H["request_approval"]
    H --> P{"approved"}
    P -- "yes" --> Q
    P -- "no" --> R["finish_rejected"]
    B --> E["END"]
    Q --> E
    R --> E
```

| LangGraph 对象 | 本实验内容 |
| --- | --- |
| State | PR、CI、环境、准入结论、审批、Job、状态 |
| Node | 读取、判断、审批、入队、阻止、拒绝 |
| Conditional Edge | 根据准入状态和审批结果路由 |
| Checkpointer | 保存每个 Super-step |
| `thread_id` | 找到同一次测试部署申请 |


### 7.3 定义 State 与节点

节点只返回自己修改的字段。审批节点首次执行时调用 `interrupt()` 并暂停；使用相同 `thread_id` 恢复后，`interrupt()` 返回外部注入的审批对象。


In [10]:
from typing import TypedDict
from langgraph.types import interrupt


class DeploymentState(TypedDict, total=False):
    pr_id: str
    pull_request: dict
    ci_report: dict
    environment: dict
    decision: dict
    approval: dict
    outcome: dict
    status: str


def load_evidence_node(state: DeploymentState) -> DeploymentState:
    current_pr = PULL_REQUESTS[state["pr_id"]]
    return {
        "pull_request": current_pr.model_dump(),
        "ci_report": CI_REPORTS[current_pr.ci_report_id].model_dump(),
        "environment": ENVIRONMENTS[current_pr.target_env].model_dump(),
        "status": "evidence_loaded",
    }


def assess_node(state: DeploymentState) -> DeploymentState:
    decision = assess_readiness(
        PullRequest.model_validate(state["pull_request"]),
        CIReport.model_validate(state["ci_report"]),
        EnvironmentState.model_validate(state["environment"]),
    )
    return {"decision": decision.model_dump(), "status": "decision_ready"}


def approval_node(state: DeploymentState) -> DeploymentState:
    decision = ReadinessDecision.model_validate(state["decision"])
    response = interrupt({
        "question": "是否批准该数据库迁移进入共享测试环境？",
        "decision": decision.model_dump(),
    })
    approval = ApprovalDecision.model_validate(response)
    return {"approval": approval.model_dump(), "status": "approval_received"}


In [11]:
LANGGRAPH_DEPLOYMENT_JOBS: dict[str, dict] = {}


def queue_job_node(state: DeploymentState) -> DeploymentState:
    approval_data = state.get("approval")
    approval = (
        ApprovalDecision.model_validate(approval_data)
        if approval_data
        else None
    )
    job = queue_test_deployment(
        LANGGRAPH_DEPLOYMENT_JOBS,
        PullRequest.model_validate(state["pull_request"]),
        ReadinessDecision.model_validate(state["decision"]),
        approval,
    )
    outcome = DeploymentOutcome(status="queued", job=job)
    return {"outcome": outcome.model_dump(), "status": "queued"}


def finish_blocked_node(state: DeploymentState) -> DeploymentState:
    decision = ReadinessDecision.model_validate(state["decision"])
    outcome = DeploymentOutcome(status="blocked", reasons=decision.reasons)
    return {"outcome": outcome.model_dump(), "status": "blocked"}


def finish_rejected_node(state: DeploymentState) -> DeploymentState:
    outcome = DeploymentOutcome(
        status="rejected",
        reasons=["Team Leader 拒绝测试部署申请"],
    )
    return {"outcome": outcome.model_dump(), "status": "rejected"}


def route_after_assessment(state: DeploymentState) -> str:
    return ReadinessDecision.model_validate(state["decision"]).status


def route_after_approval(state: DeploymentState) -> str:
    approval = ApprovalDecision.model_validate(state["approval"])
    return "queue" if approval.approved else "reject"


### 7.4 编译 Graph

`StateGraph` 描述节点和边，`compile(checkpointer=...)` 生成运行时。教学使用 `InMemorySaver`，因此关闭 Kernel 后状态消失；生产环境应换成数据库型 Checkpointer。[4]


In [12]:
from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import InMemorySaver


builder = StateGraph(DeploymentState)
builder.add_node("load_evidence", load_evidence_node)
builder.add_node("assess", assess_node)
builder.add_node("approval", approval_node)
builder.add_node("queue", queue_job_node)
builder.add_node("blocked", finish_blocked_node)
builder.add_node("reject", finish_rejected_node)

builder.add_edge(START, "load_evidence")
builder.add_edge("load_evidence", "assess")
builder.add_conditional_edges(
    "assess",
    route_after_assessment,
    {
        "blocked": "blocked",
        "ready": "queue",
        "approval_required": "approval",
    },
)
builder.add_conditional_edges(
    "approval",
    route_after_approval,
    {"queue": "queue", "reject": "reject"},
)
builder.add_edge("blocked", END)
builder.add_edge("queue", END)
builder.add_edge("reject", END)

deployment_graph = builder.compile(checkpointer=InMemorySaver())
print("Graph 编译完成")


Graph 编译完成


### 7.5 首次运行：审批前暂停

首次调用应停在审批节点。此时 State 已包含准入结论，部署队列必须为空。


In [13]:
LANGGRAPH_DEPLOYMENT_JOBS.clear()
graph_config = {
    "configurable": {"thread_id": "test-deploy-PR-1842-a1b2c3d4"}
}

first_graph_result = deployment_graph.invoke(
    {"pr_id": PR_ID},
    config=graph_config,
)

print("当前状态：", first_graph_result["status"])
print("中断请求：", first_graph_result.get("__interrupt__"))
print("部署队列：", LANGGRAPH_DEPLOYMENT_JOBS)

assert first_graph_result["status"] == "decision_ready"
assert first_graph_result.get("__interrupt__")
assert LANGGRAPH_DEPLOYMENT_JOBS == {}


当前状态： decision_ready
中断请求： [Interrupt(value={'question': '是否批准该数据库迁移进入共享测试环境？', 'decision': {'status': 'approval_required', 'reasons': ['变更包含数据库迁移，共享环境数据可能受影响'], 'requires_approval': True, 'recommended_action': 'Team Leader 批准后创建测试部署任务'}}, id='b269ff0909e5ca62b7ccd31eb6bb9c6c')]
部署队列： {}


### 7.6 恢复运行：批准后只入队一次

恢复必须使用相同 `thread_id`。`Command(resume=...)` 的值成为审批节点中 `interrupt()` 的返回值。随后流程进入独立的入队节点。


In [14]:
from langgraph.types import Command


resumed_graph_result = deployment_graph.invoke(
    Command(
        resume=ApprovalDecision(
            approved=True,
            actor="team-leader-wang",
            comment="已确认迁移仅影响共享测试环境",
        ).model_dump()
    ),
    config=graph_config,
)

print("最终状态：", resumed_graph_result["status"])
print("部署任务：", resumed_graph_result["outcome"]["job"])

assert resumed_graph_result["status"] == "queued"
assert len(LANGGRAPH_DEPLOYMENT_JOBS) == 1
assert resumed_graph_result["outcome"]["job"]["approved_by"] == "team-leader-wang"


# 使用新的 thread_id 验证拒绝分支；已有队列数量不能增加。
reject_config = {
    "configurable": {"thread_id": "test-deploy-PR-1842-rejected"}
}
deployment_graph.invoke({"pr_id": PR_ID}, config=reject_config)
rejected_graph_result = deployment_graph.invoke(
    Command(
        resume=ApprovalDecision(
            approved=False,
            actor="team-leader-wang",
            comment="迁移脚本缺少回滚验证",
        ).model_dump()
    ),
    config=reject_config,
)

print("拒绝状态：", rejected_graph_result["status"])
assert rejected_graph_result["status"] == "rejected"
assert len(LANGGRAPH_DEPLOYMENT_JOBS) == 1


最终状态： queued
部署任务： {'job_id': 'JOB-A1B2C3D4', 'idempotency_key': 'test-deploy:PR-1842:a1b2c3d4:integration-test-2', 'pr_id': 'PR-1842', 'commit_sha': 'a1b2c3d4', 'target_env': 'integration-test-2', 'status': 'queued', 'approved_by': 'team-leader-wang'}
拒绝状态： rejected


In [15]:
# State History 按时间倒序返回；这里反转后展示业务状态的推进过程。
graph_history = [
    {
        "status": snapshot.values.get("status"),
        "next": list(snapshot.next),
    }
    for snapshot in deployment_graph.get_state_history(graph_config)
]

for item in reversed(graph_history):
    print(item)

assert any(item["next"] == ["approval"] for item in graph_history)
assert graph_history[0]["status"] == "queued"


{'status': None, 'next': ['__start__']}
{'status': None, 'next': ['load_evidence']}
{'status': 'evidence_loaded', 'next': ['assess']}
{'status': 'decision_ready', 'next': ['approval']}
{'status': 'approval_received', 'next': ['queue']}
{'status': 'queued', 'next': []}


State History 记录每个 Super-step 后的状态和下一节点。恢复依赖 Checkpoint，不依赖模型复述历史。

审批节点恢复时会从函数开头重新执行。副作用应放在审批后的独立节点中，并使用业务幂等键。这样即使入队节点因超时被重试，队列也只保留一条任务。[5]


### 7.7 可选真实实验：模型解释风险，策略控制路由

模型读取同一份证据并生成 `ReadinessDecision`，本地 Pydantic 负责字段校验。真正的 Graph 路由继续使用 `assess_readiness()` 的结果，避免模型绕过 CI、环境和审批规则。


In [16]:
from langchain_openai import ChatOpenAI
from pydantic import ValidationError


langgraph_live_decision = None

if RUN_LIVE_LLM and LLM_API_KEY:
    langgraph_llm = ChatOpenAI(
        model=LLM_MODEL,
        api_key=LLM_API_KEY,
        base_url=LLM_BASE_URL,
        temperature=0,
    ).bind(response_format={"type": "json_object"})

    evidence = {
        "pull_request": PULL_REQUESTS[PR_ID].model_dump(),
        "ci_report": CI_REPORTS["CI-9001"].model_dump(),
        "environment": ENVIRONMENTS["integration-test-2"].model_dump(),
    }
    prompt = (
        "你是共享测试环境准入分析器，只返回 JSON。"
        "输出字段为 status、reasons、requires_approval、recommended_action。"
        "数据库迁移需要人工批准。证据："
        f"{evidence}"
    )

    for attempt in range(1, 4):
        message = langgraph_llm.invoke(prompt)
        try:
            langgraph_live_decision = ReadinessDecision.model_validate_json(
                message.content
            )
            print(f"第 {attempt} 次输出通过 Schema 校验")
            break
        except ValidationError as exc:
            prompt += f"\n上次输出校验失败，请修正：{exc.errors()}"

    if langgraph_live_decision is None:
        raise RuntimeError("模型连续三次未返回有效 ReadinessDecision")
    print(langgraph_live_decision.model_dump())
    assert langgraph_live_decision.status == baseline.status
else:
    print("已跳过真实 LangGraph 模型节点；设置 RUN_LIVE_LLM=1 后执行。")


已跳过真实 LangGraph 模型节点；设置 RUN_LIVE_LLM=1 后执行。


### 7.8 本框架解决了什么，还缺什么

LangGraph 已显式表达分支、审批、恢复和 State History。上线前仍需补齐：

1. 数据库型 Checkpointer 和状态版本迁移。
2. `thread_id` 与租户、用户、PR 的授权绑定。
3. CI/CD API 回执查询与未知结果处理。
4. 节点输入输出脱敏和 Tracing 保留期限。
5. Runner 执行结果的异步回调流程。


## 8 Microsoft Agent Framework：用 Executor 与请求响应组织流程

### 8.1 本节目标

本节继续实现同一条准入流程，重点观察两层抽象：

1. Agent 处理需要模型参与的消息和工具循环。
2. Workflow 用 Executor、Edge 和外部 Response 控制固定业务阶段。

离线实验使用确定性 Executor，完整验证暂停审批和入队。可选真实实验再把证据分析换成两个 Agent，避免把模型质量与 Workflow 机制混在一起。[6][8]


### 8.2 核心对象与案例映射

| 对象 | 职责 | 本实验映射 |
| --- | --- | --- |
| Agent | 模型驱动的消息与工具循环 | 证据分析和准入建议 |
| Executor | 处理指定消息类型 | 准入判断、审批、入队 |
| Edge | 连接 Executor，可带条件 | 审批、直通和阻止分支 |
| Workflow | 调度消息和 Executor | 完整测试部署申请 |
| `request_info` | 发出外部请求并暂停 | 请求 Team Leader 决策 |
| `response_handler` | 接收同一请求的响应 | 恢复审批流程 |
| Checkpoint | 保存 Workflow 状态 | 生产恢复基础 [7] |


### 8.3 定义消息和 Executor

Workflow 中的每条消息都表示一个已完成阶段。`AdmissionExecutor` 读取证据并生成准入结论；`ApprovalExecutor` 只处理需要审批的结论；`FinalizeExecutor` 负责阻止、拒绝或幂等入队。


In [17]:
from dataclasses import dataclass


@dataclass
class MAFDeploymentInput:
    pr_id: str


@dataclass
class MAFDecisionMessage:
    pull_request: PullRequest
    decision: ReadinessDecision


@dataclass
class MAFApprovalRequest:
    pull_request: PullRequest
    decision: ReadinessDecision


@dataclass
class MAFApprovalResponse:
    approved: bool
    actor: str
    comment: str


@dataclass
class MAFApprovedMessage:
    pull_request: PullRequest
    decision: ReadinessDecision
    approval: MAFApprovalResponse


In [18]:
from agent_framework import (
    Executor,
    WorkflowContext,
    handler,
    response_handler,
)


class AdmissionExecutor(Executor):
    @handler
    async def assess(
        self,
        message: MAFDeploymentInput,
        ctx: WorkflowContext[MAFDecisionMessage],
    ) -> None:
        current_pr = PULL_REQUESTS[message.pr_id]
        decision = assess_readiness(
            current_pr,
            CI_REPORTS[current_pr.ci_report_id],
            ENVIRONMENTS[current_pr.target_env],
        )
        await ctx.send_message(MAFDecisionMessage(current_pr, decision))


class ApprovalExecutor(Executor):
    def __init__(self):
        super().__init__(id="approval-executor")

    @handler
    async def request_approval(
        self,
        message: MAFDecisionMessage,
        ctx: WorkflowContext,
    ) -> None:
        await ctx.request_info(
            MAFApprovalRequest(message.pull_request, message.decision),
            MAFApprovalResponse,
            request_id=f"approval-{message.pull_request.pr_id}",
        )

    @response_handler
    async def receive_approval(
        self,
        original_request: MAFApprovalRequest,
        response: MAFApprovalResponse,
        ctx: WorkflowContext[MAFApprovedMessage],
    ) -> None:
        await ctx.send_message(
            MAFApprovedMessage(
                original_request.pull_request,
                original_request.decision,
                response,
            )
        )


In [19]:
MAF_DEPLOYMENT_JOBS: dict[str, dict] = {}


class FinalizeExecutor(Executor):
    @handler
    async def finalize(
        self,
        message: MAFDecisionMessage | MAFApprovedMessage,
        ctx: WorkflowContext[None, DeploymentOutcome],
    ) -> None:
        if isinstance(message, MAFDecisionMessage):
            if message.decision.status == "blocked":
                outcome = DeploymentOutcome(
                    status="blocked",
                    reasons=message.decision.reasons,
                )
            else:
                job = queue_test_deployment(
                    MAF_DEPLOYMENT_JOBS,
                    message.pull_request,
                    message.decision,
                )
                outcome = DeploymentOutcome(status="queued", job=job)
        elif message.approval.approved:
            approval = ApprovalDecision(
                approved=True,
                actor=message.approval.actor,
                comment=message.approval.comment,
            )
            job = queue_test_deployment(
                MAF_DEPLOYMENT_JOBS,
                message.pull_request,
                message.decision,
                approval,
            )
            outcome = DeploymentOutcome(status="queued", job=job)
        else:
            outcome = DeploymentOutcome(
                status="rejected",
                reasons=["Team Leader 拒绝测试部署申请"],
            )

        await ctx.yield_output(outcome)


### 8.4 构建并运行 Workflow

条件 Edge 根据 `ReadinessDecision` 路由：

- `approval_required`：等待外部 Response。
- `ready` 或 `blocked`：直接进入 Finalize，由业务代码决定入队或结束。

首次运行应进入 `IDLE_WITH_PENDING_REQUESTS`。这表示 Workflow 暂无可执行节点，同时保存了一条等待响应的请求。


In [20]:
from agent_framework import WorkflowBuilder, WorkflowRunState


admission_executor = AdmissionExecutor(id="admission-executor")
approval_executor = ApprovalExecutor()
finalize_executor = FinalizeExecutor(id="finalize-executor")

maf_workflow = (
    WorkflowBuilder(
        start_executor=admission_executor,
        output_from=[finalize_executor],
    )
    .add_edge(
        admission_executor,
        approval_executor,
        condition=lambda message: message.decision.requires_approval,
    )
    .add_edge(
        admission_executor,
        finalize_executor,
        condition=lambda message: not message.decision.requires_approval,
    )
    .add_edge(approval_executor, finalize_executor)
    .build()
)


async def run_maf_offline():
    MAF_DEPLOYMENT_JOBS.clear()
    first = await maf_workflow.run(MAFDeploymentInput(PR_ID))
    requests = first.get_request_info_events()

    print("首次状态：", first.get_final_state())
    print("等待请求：", [event.request_id for event in requests])
    print("审批前队列：", MAF_DEPLOYMENT_JOBS)

    assert first.get_final_state() == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS
    assert first.get_outputs() == []
    assert len(requests) == 1
    assert MAF_DEPLOYMENT_JOBS == {}

    request_id = requests[0].request_id
    second = await maf_workflow.run(
        responses={
            request_id: MAFApprovalResponse(
                approved=True,
                actor="team-leader-wang",
                comment="已确认迁移仅影响共享测试环境",
            )
        }
    )
    outcome = second.get_outputs()[0]
    print("恢复状态：", second.get_final_state())
    print("最终输出：", outcome.model_dump())

    assert second.get_final_state() == WorkflowRunState.IDLE
    assert outcome.status == "queued"
    assert len(MAF_DEPLOYMENT_JOBS) == 1
    return first, second


maf_first_result, maf_second_result = await run_maf_offline()


首次状态： WorkflowRunState.IDLE_WITH_PENDING_REQUESTS
等待请求： ['approval-PR-1842']
审批前队列： {}
恢复状态： WorkflowRunState.IDLE
最终输出： {'status': 'queued', 'reasons': [], 'job': {'job_id': 'JOB-A1B2C3D4', 'idempotency_key': 'test-deploy:PR-1842:a1b2c3d4:integration-test-2', 'pr_id': 'PR-1842', 'commit_sha': 'a1b2c3d4', 'target_env': 'integration-test-2', 'status': 'queued', 'approved_by': 'team-leader-wang'}}


Microsoft Agent Framework 使用 `request_info()`、`request_id` 和类型化 Response 表达暂停恢复。原始请求随 Workflow 状态保存，恢复处理器同时获得原请求和新响应。

真实审批系统需要持久化 `request_id`、PR、Commit、租户和审批单号的映射。入队仍由 `queue_test_deployment()` 检查批准状态和幂等键。[7]


### 8.5 可选真实实验：Agent 负责分析，Workflow 固定顺序

真实实验串联两个 Agent：

1. `EvidenceAgent` 必须调用工具读取三份证据。
2. `AdmissionAgent` 把证据摘要转换为明确准入建议。

这段实验只观察模型 Agent 的组合，不创建部署任务。写操作继续留在确定性 Executor。


In [21]:
from agent_framework import tool
from agent_framework.openai import OpenAIChatCompletionClient


maf_live_output = None

if RUN_LIVE_LLM and LLM_API_KEY:
    @tool(approval_mode="never_require")
    def get_maf_deployment_evidence() -> dict:
        '''读取固定实验 PR 的 PR、CI 和环境证据。'''
        current_pr = PULL_REQUESTS[PR_ID]
        return {
            "pull_request": current_pr.model_dump(),
            "ci_report": CI_REPORTS[current_pr.ci_report_id].model_dump(),
            "environment": ENVIRONMENTS[current_pr.target_env].model_dump(),
        }

    maf_client = OpenAIChatCompletionClient(
        model=LLM_MODEL,
        api_key=LLM_API_KEY,
        base_url=LLM_BASE_URL,
    )
    evidence_agent = maf_client.as_agent(
        name="EvidenceAgent",
        instructions="必须调用工具读取证据，并原样总结关键字段。",
        tools=get_maf_deployment_evidence,
    )
    admission_agent = maf_client.as_agent(
        name="AdmissionAgent",
        instructions=(
            "根据上游证据给出准入建议。数据库迁移输出 "
            "DECISION=APPROVAL_REQUIRED；其他失败输出 DECISION=BLOCKED。"
        ),
    )
    maf_live_workflow = (
        WorkflowBuilder(
            start_executor=evidence_agent,
            output_from=[admission_agent],
        )
        .add_chain([evidence_agent, admission_agent])
        .build()
    )
    maf_live_result = await maf_live_workflow.run(
        f"评估 {PR_ID} 的共享测试环境部署申请。"
    )
    maf_live_output = str(maf_live_result.get_outputs()[0])
    print(maf_live_output)
    assert "APPROVAL_REQUIRED" in maf_live_output
else:
    print("已跳过真实 Microsoft Agent Framework 调用。")


已跳过真实 Microsoft Agent Framework 调用。


Workflow 固定两个 Agent 的执行顺序，Agent 内部仍由模型选择工具和生成消息。准入、审批和入队保持确定性，可以减少多 Agent 对关键业务状态的影响。

Microsoft Agent Framework 支持 OpenAI-compatible Endpoint。[16] 基础 Chat、Tool Calling、Structured Outputs、流式事件和 Usage 字段仍需分别验证。


### 8.6 本框架解决了什么，还缺什么

Microsoft Agent Framework 已表达类型化消息、条件 Edge、人工请求和 Agent + Workflow 组合。上线前仍需补齐：

1. 持久化 Checkpoint 与历史状态升级回放。
2. 审批平台回调的鉴权、防重放和超时处理。
3. Workflow Event 到统一 Tracing 协议的映射。
4. CI/CD API 的测试环境专用服务账号。
5. 多 Agent 重复上下文带来的 Token 和延迟预算。


## 9 横向对照：同一条测试部署准入流程

### 9.1 控制流与状态

| 维度 | PydanticAI | LangGraph | Microsoft Agent Framework |
| --- | --- | --- | --- |
| 主要抽象 | Agent | StateGraph | Agent + Executor + Workflow |
| 本章实现范围 | 证据读取与类型化结论 | 完整准入、审批、恢复、入队 | 完整准入、请求响应、入队 |
| 默认控制方式 | 模型驱动工具循环 | 图和条件边 | Agent 模型驱动，Workflow 显式驱动 |
| 状态位置 | 消息、依赖、外部持久化 | 显式 State + Checkpointer | 消息、Executor、Workflow Checkpoint |
| 人工中断 | 组合外部运行时 | `interrupt` + `Command` | `request_info` + Response |
| 副作用安全 | 工具层业务幂等 | 独立 Node + 业务幂等 | 独立 Executor + 业务幂等 |
| 最适合学习的能力 | 类型契约与依赖注入 | 状态、路由和恢复 | Agent 与 Workflow 组合 |


### 9.2 三种运行轨迹

```text
PydanticAI
Prompt -> ToolCall(evidence) -> ToolReturn -> ReadinessDecision

LangGraph
START -> load_evidence -> assess -> interrupt
      -> resume -> queue_job -> END

Microsoft Agent Framework
AdmissionExecutor -> Approval Request -> external Response
                  -> FinalizeExecutor -> DeploymentOutcome
```

PydanticAI 的轨迹围绕模型消息展开；LangGraph 围绕 State Super-step 展开；Microsoft Agent Framework 围绕 Executor 消息和 Workflow Event 展开。生产 Tracing 需要把这些事件映射为团队统一的 `run_id`、`step_id`、`action_id` 和 `parent_id`。


In [22]:
from importlib.metadata import version


installed_versions = {
    "pydantic-ai": version("pydantic-ai"),
    "langgraph": version("langgraph"),
    "langchain-openai": version("langchain-openai"),
    "agent-framework-core": version("agent-framework-core"),
    "agent-framework-openai": version("agent-framework-openai"),
    "agent-framework-orchestrations": version("agent-framework-orchestrations"),
}
print(installed_versions)


{'pydantic-ai': '2.12.0', 'langgraph': '1.2.9', 'langchain-openai': '1.3.5', 'agent-framework-core': '1.11.0', 'agent-framework-openai': '1.10.1', 'agent-framework-orchestrations': '1.0.0'}


版本快照属于实验结果的一部分。框架 API、Provider 参数、Checkpoint Schema 和事件类型都会变化。生产项目应同时锁定：

1. Python 依赖版本与 Lockfile。
2. 模型 ID 和 Endpoint。
3. Tool Schema 与业务数据协议版本。
4. 固定回归任务和历史 Checkpoint 兼容测试。

只锁 Python 包不能阻止模型服务端行为变化。


## 10 How：如何选择框架

### 10.1 决策顺序

```mermaid
flowchart TD
    A["任务是否只有一次或短工具循环"] -->|"是"| B["普通 SDK 或 PydanticAI"]
    A -->|"否"| C["流程是否有固定阶段和恢复要求"]
    C -->|"是"| D["LangGraph 或 Workflow Runtime"]
    C -->|"否"| E["轻量 Agent SDK + 自有 Harness"]
    D --> F["是否需要多个独立 Agent 角色"]
    F -->|"否"| G["单 Agent 图或 Workflow"]
    F -->|"是"| H["混合 Workflow + Agent"]
    H --> I["评估 Microsoft Agent Framework 等方案"]
```

决策从任务控制流开始，再看生态和团队偏好。不要先选框架，再把简单函数包装成多个 Agent。


### 10.2 场景建议

| 场景 | 优先方案 | 原因 |
| --- | --- | --- |
| API 服务中增加类型化模型能力 | PydanticAI | Pydantic 契约、依赖注入和测试路径直接 |
| 长流程、循环、审批和故障恢复 | LangGraph | State、Edge、Interrupt 和 Checkpoint 显式 |
| Microsoft 生态中的 Agent + Workflow | Microsoft Agent Framework | Agent 与 Executor 可统一编排 |
| 五个固定步骤、无模型自主决策 | 普通 Python Workflow | 调试成本最低 |
| 低代码知识库和运营后台 | Dify / Coze Studio 等平台 | 平台能力比 Python 抽象更重要 |
| 高风险写操作 | 任一框架 + 独立 Policy/HITL/幂等 | 框架本身不构成安全边界 |


### 10.3 生产选型用实验评分，不用功能清单

建议为候选框架运行同一组 PoC：

| 指标 | 观察方法 |
| --- | --- |
| 正确率 | 固定任务集通过率 |
| 可恢复性 | 在模型调用前后、工具执行前后和持久化前后注入崩溃 |
| 副作用安全 | 重试后外部记录是否重复 |
| Provider 兼容 | Chat、Tool Calling、JSON、流式、Usage 分别验证 |
| 可观测性 | 是否可关联模型、工具、节点、审批和最终结果 |
| 升级成本 | 新版本能否读取旧状态和通过回归任务 |
| 延迟与成本 | 每任务模型调用数、Token、P95 延迟 |
| 团队维护成本 | 自定义扩展、调试、值班和文档成熟度 |

框架官网列出的“支持 HITL”无法替代崩溃恢复实验。生产价值取决于具体存储、状态协议和故障窗口。


## 11 生产常见坑与 Trade-off

### 11.1 Provider 兼容是能力矩阵

使用 OpenAI-compatible API 时，至少分别验证：

1. 基础 Chat Completion。
2. 单工具与并列 Tool Call。
3. `tool_choice` 的 `auto`、指定和强制模式。
4. JSON Mode 与 JSON Schema Structured Outputs。
5. 流式 Tool Call 参数拼接。
6. Usage、推理内容和错误字段。

本章真实实验已经遇到一个具体差异：DeepSeek 推理模式拒绝框架生成的强制 `tool_choice`。改为 Prompted Output 可以绕开该路径，但严格程度和重试行为也随之变化。


### 11.2 框架重试与副作用

本章的入队调用可能出现一个典型故障窗口：

```text
CI/CD 已创建 Job -> 网络断开 -> Agent 没有保存成功结果
```

恢复后直接重试会创建重复 Job。处理步骤如下：

1. 使用稳定幂等键：`test-deploy:{pr_id}:{commit_sha}:{target_env}`。
2. CI/CD API 支持幂等键时直接传入。
3. API 不支持时，在本地 Outbox / Receipt Store 建立唯一约束。
4. 超时后先按幂等键查询，再决定是否重试。
5. 审批放在入队之前，Runner 结果验收放在入队之后。

Checkpoint 保存流程进度。业务系统的唯一约束和回执记录负责副作用安全。


### 11.3 状态、序列化与升级

| 风险 | 典型现象 | 对策 |
| --- | --- | --- |
| 框架对象进入业务数据 | 离开框架后无法复用 | 边界处转换为领域模型 |
| State 无限增长 | Checkpoint 慢、存储膨胀 | 大内容落 Artifact，只存引用 |
| Schema 变化 | 旧 Checkpoint 无法恢复 | 状态版本号和迁移函数 |
| Executor / Node 重命名 | 恢复时找不到目标 | 稳定 ID、升级回放测试 |
| 多租户混用 thread/session | 用户读到其他会话 | 租户键、授权校验、加密和审计 |
| 内存 Checkpointer 上线 | 重启后任务消失 | 数据库型持久化和备份恢复演练 |


### 11.4 隐式行为与调试

框架可能自动增加 System Prompt、输出工具、重试消息、工具错误说明和历史消息。调试时应保存或观测以下原始事实：

1. 发给模型的最终消息和 Tool Schema。
2. 模型原始响应、finish reason 和 Usage。
3. 每次工具参数、结果、耗时和错误。
4. 每个节点的输入 State、增量更新和下一节点。
5. 自动重试次数及触发原因。

Tracing 中可能包含源码、用户数据、工具结果和 Prompt。生产环境需要字段脱敏、分级访问和保留期限。第 12 章会完整处理这部分。


### 11.5 多 Agent 的成本

假设三个 Agent 都接收 20,000 Token 背景材料，每个 Agent 再运行两轮，输入成本会接近单 Agent 的数倍。并行只能降低墙钟时间，不能降低总 Token。

拆分多 Agent 前检查：

- 角色是否拥有不同权限或工具。
- 子任务是否可以使用更小的上下文。
- 并行是否带来实际延迟收益。
- 交接协议是否比共享全部对话更短。
- 汇总 Agent 是否有独立证据验证结果。

只有角色名称不同、上下文和工具完全相同的多 Agent，通常只会增加调用次数和故障点。


## 12 When：什么时候使用、什么时候克制

### 12.1 使用框架

满足以下任意两项时，框架通常开始产生价值：

1. 存在三步以上的模型与工具交互。
2. 任务需要暂停、恢复或跨进程运行。
3. 存在条件分支、循环、并行或多 Agent。
4. 多个团队需要统一 Agent、工具和事件接口。
5. 需要替换模型供应商并保持业务层稳定。

### 12.2 保持普通代码

以下任务优先使用函数、队列和状态机：

1. 路径完全固定且没有模型决策。
2. 一次模型调用即可得到结果。
3. 任务延迟和依赖体积要求严格。
4. 团队还没有稳定的输入输出协议、评测集和故障处理规则。


## 13 与产研协作助手的关系

第 13 章可以复用本章的工程方法：

1. 先定义真实角色、权限、业务状态和工具边界。
2. 用 Pydantic 模型约束需求、任务、测试结果、风险和审批协议。
3. 用显式 Workflow 管理“读取资料 → 分析 → 查询系统 → 生成建议 → 人工确认 → 写入”。
4. 把局部分析交给模型，把硬规则、权限、审批和写操作放在确定性节点。
5. 为外部写操作设计幂等键、回执查询和未知结果恢复。

最终框架选择由 Capstone PoC 的正确率、恢复、成本和维护数据决定。


## 14 本章小结

1. Agent 框架由编程模型、运行时和集成层组成。
2. 框架选型先判断控制权、状态归属、暂停恢复和副作用边界。
3. 测试部署包含准入、审批、任务入队和 Runner 执行，四个动作需要分别定义。
4. Agent 可以读取证据和创建测试任务，不应持有生产凭据。
5. PydanticAI 适合类型化 Agent 和 Python 依赖注入。
6. LangGraph 适合显式状态、条件路由、中断和恢复。
7. Microsoft Agent Framework 适合组合模型 Agent 与确定性 Workflow。
8. Checkpoint 不负责副作用去重，入队操作必须使用业务幂等键。
9. 本地模拟器可以验证流程语义；接入生产时再替换 PR、CI、审批和队列适配器。


## 15 下一步

框架解决了 Agent 的定义、控制流、状态和组合。生产系统还需要回答四个问题：

1. 如何持续判断 Agent 是否正确。
2. 如何追踪模型、工具、节点和审批。
3. 如何阻止越权、注入和危险输出。
4. 如何在高风险动作前要求人工确认。

第 12 章将学习 Evals、Tracing、Guardrails 与 Human-in-the-loop，并把这些能力放到同一条生产控制链中。


## 参考文献

[1] ANTHROPIC. Building effective agents[EB/OL]. (2024-12-19)[2026-07-17]. https://www.anthropic.com/engineering/building-effective-agents.

[2] PYDANTIC. Pydantic AI Documentation[EB/OL]. [2026-07-17]. https://ai.pydantic.dev/.

[3] PYDANTIC. Durable Execution Overview[EB/OL]. [2026-07-17]. https://ai.pydantic.dev/durable_execution/overview/.

[4] LANGCHAIN. LangGraph Persistence[EB/OL]. [2026-07-17]. https://docs.langchain.com/oss/python/langgraph/persistence.

[5] LANGCHAIN. LangGraph Interrupts[EB/OL]. [2026-07-17]. https://docs.langchain.com/oss/python/langgraph/interrupts.

[6] MICROSOFT. Microsoft Agent Framework Overview[EB/OL]. (2026-07-10)[2026-07-17]. https://learn.microsoft.com/en-us/agent-framework/overview/.

[7] MICROSOFT. Workflow Checkpoints[EB/OL]. [2026-07-17]. https://learn.microsoft.com/en-us/agent-framework/workflows/checkpoints.

[8] MICROSOFT. Using AI Agents in Workflows[EB/OL]. [2026-07-17]. https://learn.microsoft.com/en-us/agent-framework/workflows/advanced/agent-executor.

[9] OPENAI. OpenAI Agents SDK for Python[EB/OL]. [2026-07-17]. https://openai.github.io/openai-agents-python/.

[10] GOOGLE. Agent Development Kit for Python[EB/OL]. [2026-07-17]. https://github.com/google/adk-python.

[11] STRANDS AGENTS. Strands Agents SDK for Python[EB/OL]. [2026-07-17]. https://github.com/strands-agents/sdk-python.

[12] LLAMAINDEX. LlamaIndex[EB/OL]. [2026-07-17]. https://github.com/run-llama/llama_index.

[13] CREWAI. CrewAI[EB/OL]. [2026-07-17]. https://github.com/crewAIInc/crewAI.

[14] LANGGENIUS. Dify[EB/OL]. [2026-07-17]. https://github.com/langgenius/dify.

[15] COZEDEV. Coze Studio[EB/OL]. [2026-07-17]. https://github.com/coze-dev/coze-studio.

[16] MICROSOFT. Using OpenAI-compatible endpoints[EB/OL]. [2026-07-17]. https://learn.microsoft.com/en-us/agent-framework/integrations/openai-endpoints.
